The implementation of this notebook is largely based on [the work by Vera Neplenbroek](https://github.com/Veranep/crosslingualdetoxdebias).

In [3]:
!pip install peft

In [4]:
from datasets import load_dataset
from peft import get_peft_model
from peft import prepare_model_for_kbit_training
from peft import LoraConfig
from peft import TaskType

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [5]:
HF_MODEL_NAME = "Qwen/Qwen2.5-0.5B"
NUM_TRAINING_SAMPLES = 1000 # up to 94966. Using an A100, training takes ~1.30 hours. On Colab's T4, training on 50k samples take approximately 4 hours.
NUM_PANDA_VAL_SAMPLES = 250 # up to 10551. These are the eval samples used during training.

In [6]:
def tokenize_panda(element):
    outputs = tokenizer(
        element["perturbed"], # We train on the perturbed (against the stereotypes) samples of the dataset
        truncation=True,
        max_length=512,
        padding="max_length",
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for input_ids in outputs["input_ids"]:
        input_batch.append(input_ids)
    return {"input_ids": input_batch}

In [7]:
print("Initialising Model")
peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        target_modules="all-linear",
        bias="none",
        r=64,
        lora_alpha=16,
        lora_dropout=0.1,
    )
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME,
    device_map="auto",
    )

Initialising Model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [8]:
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1
model.enable_input_require_grads()

print("Loading Panda Dataset")
raw_datasets = load_dataset("facebook/panda")
valid_name = "validation"

print("Preparing Panda Dataset")
tokenized_datasets = raw_datasets.map(
    tokenize_panda,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

Loading Panda Dataset


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/194M [00:00<?, ?B/s]

valid.jsonl:   0%|          | 0.00/21.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/94966 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10551 [00:00<?, ? examples/s]

Preparing Panda Dataset


Map:   0%|          | 0/94966 [00:00<?, ? examples/s]

Map:   0%|          | 0/10551 [00:00<?, ? examples/s]

In [9]:
training_args = TrainingArguments(
    output_dir=f"{HF_MODEL_NAME.split('/')[-1]}_lora_bias_sft_model",
    num_train_epochs=1,
    save_total_limit=5,
    eval_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    warmup_steps=150,
    weight_decay=0.001,
    metric_for_best_model="eval_loss",
    fp16=True,
    remove_unused_columns=False,
    logging_steps=1000,
    eval_steps=1000,
    save_steps=1000,
    save_strategy="steps",
    learning_rate=3e-4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    load_best_model_at_end=True,
    seed=42,
)
print("Getting PEFT Model ready")
model = get_peft_model(model, peft_config)

Getting PEFT Model ready


In [10]:
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(NUM_TRAINING_SAMPLES)),
    eval_dataset=tokenized_datasets[valid_name].select(range(NUM_PANDA_VAL_SAMPLES)),
    data_collator=data_collator,
)

print("START TRAINING")
trainer.train()
debiased_model = model.merge_and_unload()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


START TRAINING


Step,Training Loss,Validation Loss


In [ ]:
#### ADD YOUR HUGGINGFACE REPO_ID HERE
debiased_model.push_to_hub('repo_id')
tokenizer.push_to_hub('repo_id')